# Traffic CNN - Vehicle Detection Training

Este notebook permite entrenar el modelo Traffic CNN en Google Colab usando el dataset **vehicleDataset** de Roboflow.

## 📋 Prerrequisitos
1. Subir `vehicleDataset.zip` a tu Google Drive (carpeta raíz "Mi unidad")
2. Ejecutar las celdas en orden

## 📊 Dataset
- **Clases:** 7 tipos de vehículos (bicycle, bus, car, motorbike, rickshaw, truck, van)
- **Formato:** YOLO con coordenadas normalizadas
- **Estructura:** train/valid/test con imágenes y labels separados

## 1️⃣ Setup Inicial - Montar Google Drive

In [ ]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Crear directorio para modelos
os.makedirs('/content/drive/MyDrive/Traffic_CNN_Models', exist_ok=True)
print("✅ Drive montado y directorio de modelos creado")

## 2️⃣ Descomprimir Dataset

In [ ]:
import os

print("📂 Descomprimiendo dataset desde Drive...")
!unzip -q /content/drive/MyDrive/vehicleDataset.zip -d /content/
print("✅ Dataset extraído")

# Verificar estructura
print("\n📁 Estructura del dataset:")
!ls -lh /content/train/

# Contar imágenes
train_imgs = len([f for f in os.listdir('/content/train/images') if f.endswith('.jpg')])
valid_imgs = len([f for f in os.listdir('/content/valid/images') if f.endswith('.jpg')])
test_imgs = len([f for f in os.listdir('/content/test/images') if f.endswith('.jpg')])

print(f"\n📊 Imágenes disponibles:")
print(f"   - Train: {train_imgs}")
print(f"   - Valid: {valid_imgs}")
print(f"   - Test: {test_imgs}")

## 3️⃣ Clonar Repositorio

In [ ]:
# Clonar repositorio desde GitHub
!git clone https://github.com/carloscabani/Traffic_CNN-.git
%cd Traffic_CNN-

print("✅ Repositorio clonado")
!pwd

## 4️⃣ Instalar Dependencias

In [ ]:
# Instalar paquetes necesarios
!pip install -q torch torchvision torchaudio pillow numpy matplotlib opencv-python tqdm seaborn

print("✅ Dependencias instaladas")

# Verificar versiones
import torch
print(f"\n📦 Versiones:")
print(f"   - PyTorch: {torch.__version__}")
print(f"   - CUDA disponible: {torch.cuda.is_available()}")

## 5️⃣ Importar y Configurar

In [ ]:
import sys
import os

# Agregar src al path
sys.path.insert(0, '/content/Traffic_CNN-/src')

# Importar módulos
from dataset.dtset import TrafficFlowDataset
from models.architecture import TrafficQuantizerNet
from models.loss import TrafficLoss

print("✅ Módulos importados correctamente")

## 6️⃣ Crear Datasets

In [ ]:
# Crear dataset de entrenamiento
train_dataset = TrafficFlowDataset(
    img_dir='/content/train/images',
    label_dir='/content/train/labels',
    input_size=512,
    stride=4
)

# Crear dataset de validación
valid_dataset = TrafficFlowDataset(
    img_dir='/content/valid/images',
    label_dir='/content/valid/labels',
    input_size=512,
    stride=4
)

print(f"✅ Datasets creados:")
print(f"   - Train: {len(train_dataset)} imágenes")
print(f"   - Valid: {len(valid_dataset)} imágenes")

# Verificar un sample
sample = train_dataset[0]
print(f"\n📊 Dimensiones de un sample:")
print(f"   - Input: {sample['input'].shape}")
print(f"   - Heatmap: {sample['hm'].shape}")
print(f"   - Size: {sample['wh'].shape}")
print(f"   - Offset: {sample['reg'].shape}")

## 7️⃣ Entrenar Modelo

**Nota:** El entrenamiento puede tardar varias horas dependiendo de:
- Número de épocas configuradas (por defecto: 50)
- Tamaño del dataset (7556 imágenes de entrenamiento)
- Tipo de GPU disponible en Colab

Los checkpoints se guardarán automáticamente en Google Drive cada 5 épocas.

In [ ]:
# Cambiar al directorio del código fuente
%cd /content/Traffic_CNN-/src/train

# Ejecutar script de entrenamiento
!python train_grid_detector.py

print("\n✅ Entrenamiento completado")

### 7.1 (Opcional) Reanudar Entrenamiento

Si el entrenamiento se interrumpió, puedes reanudarlo desde el último checkpoint:

In [ ]:
# Reanudar desde el último checkpoint
!python train_grid_detector.py --resume

## 8️⃣ Verificar Modelos Guardados

In [ ]:
import os

# Listar modelos guardados
model_dir = '/content/drive/MyDrive/Traffic_CNN_Models'

print("📁 Modelos guardados en Google Drive:\n")
!ls -lh /content/drive/MyDrive/Traffic_CNN_Models/

# Verificar tamaño total
total_size = 0
for f in os.listdir(model_dir):
    if f.endswith('.pth'):
        file_path = os.path.join(model_dir, f)
        size = os.path.getsize(file_path) / (1024 * 1024)  # MB
        total_size += size

print(f"\n💾 Espacio utilizado: {total_size:.2f} MB")
print("\n✅ Los modelos están guardados permanentemente en tu Google Drive")

## 🎉 ¡Entrenamiento Completado!

### Próximos Pasos:

1. **Usar el modelo para inferencia:** Puedes cargar el modelo final (`traffic_model_final.pth`) para hacer predicciones
2. **Experimentar con hiperparámetros:** Modifica el learning rate, batch size, o número de épocas
3. **Visualizar resultados:** Usa el modelo entrenado para detectar vehículos en nuevas imágenes

### Archivos Importantes:
- **Modelo final:** `/content/drive/MyDrive/Traffic_CNN_Models/traffic_model_final.pth`
- **Checkpoints:** `/content/drive/MyDrive/Traffic_CNN_Models/traffic_model_ep*.pth`

### Dataset utilizado:
- **Clases:** bicycle (0), bus (1), car (2), motorbike (3), rickshaw (4), truck (5), van (6)
- **Formato:** YOLO con coordenadas normalizadas